<a href="https://colab.research.google.com/github/Ninagfcosta/Podcast_test/blob/main/homeguard_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""
HomeGuard Security System Simulator
Author: Janaina Hoffmann
Description: Simulador de sistema de segurança doméstica.
"""

import random
from datetime import datetime

HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

current_mode = "AWAY"

In [7]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    sensor = {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold,
        "current_value": None
    }
    return sensor

def create_alert(severity, message, sensor_id, timestamp):
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp
    }
    return alert

sensors = [
    create_sensor("MOTION_001", "Living Room", "motion"),
    create_sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    create_sensor("DOOR_001", "Front Door", "door"),
    create_sensor("SMOKE_001", "Bedroom", "smoke")
]

print(f"Initialized {len(sensors)} sensors")

Initialized 4 sensors


In [8]:
def is_abnormal_reading(sensor, reading_value):
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    elif sensor_type == "motion":
        return reading_value == True
    elif sensor_type == "door":
        return reading_value == "OPEN"
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"
    return False

def should_trigger_security_alert(sensor, reading_value, system_mode):
    if system_mode != "AWAY":
        return False
    if sensor["type"] == "motion" and reading_value == True:
        return True
    if sensor["type"] == "door" and reading_value == "OPEN":
        return True
    return False

In [9]:
def generate_reading(sensor):
    t = sensor["type"]
    if t == "temperature":
        return random.randint(30, 100)
    elif t == "motion":
        return random.choice([True, False])
    elif t == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif t == "smoke":
        return random.choice(["CLEAR", "DETECTED"])

def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")

    if should_trigger_security_alert(sensor, reading_value, system_mode):
        msg = f"SECURITY: {sensor['location']} triggered in AWAY mode!"
        alerts.append(create_alert("HIGH", msg, sensor["id"], timestamp))

    if sensor["type"] == "temperature":
        if reading_value < 35:
            alerts.append(create_alert("CRITICAL", f"Frozen pipe risk! {reading_value}°F", sensor["id"], timestamp))
        elif reading_value > 95:
            alerts.append(create_alert("HIGH", f"Equipment failure! {reading_value}°F", sensor["id"], timestamp))
        elif system_mode == "HOME" and not (65 <= reading_value <= 75):
            alerts.append(create_alert("LOW", f"Comfort: {reading_value}°F fora do range", sensor["id"], timestamp))

    if sensor["type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert("CRITICAL", "SMOKE DETECTED! Fire risk!", sensor["id"], timestamp))

    return alerts

def trigger_alert(alert):
    symbols = {"LOW": "ℹ️", "MEDIUM": "⚠️", "HIGH": "🚨", "CRITICAL": "🔥"}
    s = symbols.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {s} {alert['severity']}: {alert['message']}")

def log_event(message, timestamp=None):
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [10]:
class Sensor:

    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None

    def read(self):
        sensor_dict = {"type": self.type}
        self.current_value = generate_reading(sensor_dict)
        return self.current_value

    def isAbnormal(self):
        sensor_dict = {"type": self.type, "threshold": self.threshold}
        return is_abnormal_reading(sensor_dict, self.current_value)

    def reset(self):
        self.current_value = None

    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"

In [11]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")

    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]

    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        for sensor in sensors:
            reading = sensor.read()

            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")

            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

    print("\n" + "=" * 50)
    print("Simulation complete!")

run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 10:03:03
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Living Room triggered in AWAY mode!
[LOG] [10:03:03] Sending notification to homeowner...
[READING] Kitchen Temperature: 63°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SMOKE DETECTED! Fire risk!
[LOG] [10:03:03] Sending notification to homeowner...

Time: 10:03:03
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 32°F (Abnormal)
[ALERT!] 🔥 CRITICAL: Frozen pipe risk! 32°F
[LOG] [10:03:03] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 10:03:03
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: SECURITY: Living Room triggered in AWAY mode!
[LOG] [10:03:03] Sending notification to homeowner...
[READING] Kitchen Temperature: 91°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: SMOK